# 3. Selectors and queries

A selector is a small, composable description of *which compartments you mean*.
It is declarative: you write it once against the property vocabulary, and the
map compiles it into an integer index array.

This matters because the alternative — writing index arithmetic by hand — breaks
the moment somebody adds a stratification. A selector written before that change
keeps meaning the same thing after it.

In [ ]:
from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
vax = Property("vaccination", ("none", "one-dose", "two-dose"))

pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(vax)
)
assert pmap.size == 27

## The three resolution methods

| Method | Returns | Use for |
|---|---|---|
| `mask(sel)` | `bool` array, length `size` | Boolean arithmetic, plotting overlays |
| `select(sel)` | `int32` index array | Gathers, scatter targets, flow endpoints |
| `select_one(sel)` | `int` | Assertions that a query is unambiguous |

In [ ]:
mask = pmap.mask(state["I"])
indices = pmap.select(state["I"])

assert mask.dtype == bool and mask.shape == (pmap.size,)
assert indices.dtype.name == "int32"
assert indices.size == 9  # 3 ages x 3 vaccination levels
assert mask.sum() == indices.size

`select_one` raises unless the query matches exactly one compartment. Use it
wherever a single compartment is a modelling assumption — seeding an epidemic,
for instance — so that a later stratification fails loudly instead of silently
seeding one of nine compartments.

In [ ]:
seed = pmap.select_one(state["I"] & age["0-4"] & vax["none"])
print("seed compartment:", seed, pmap.labels()[seed])

try:
    pmap.select_one(state["I"])
except ValueError as exc:
    print(exc)

## Combining selectors

`&` is conjunction, `|` is disjunction, `~` is negation. They nest freely.

In [ ]:
unvaccinated_children = pmap.select(age[("0-4", "5-9")] & vax["none"])
assert unvaccinated_children.size == 6  # 2 ages x 3 states

not_susceptible = pmap.select(~state["S"])
assert not_susceptible.size == 18

boosted_or_old = pmap.select(vax["two-dose"] | age["10+"])
assert boosted_or_old.size == 15  # 9 + 9 - 3 overlap

### `isin` versus chained `|`

For several traits of the *same* property, prefer the `IsIn` form. It is one
node and one pass rather than a tree of disjunctions, and it reads closer to the
modelling intent.

In [ ]:
chained = pmap.select(vax["one-dose"] | vax["two-dose"])
isin = pmap.select(vax[("one-dose", "two-dose")])

assert chained.tolist() == isin.tolist()

## Everything and nothing

`Everything()` and `Nothing()` are the identity and zero of the algebra. They are
useful as defaults in code that builds selectors programmatically.

In [ ]:
from summer4 import Everything, Nothing
from functools import reduce
import operator

assert pmap.select(Everything()).size == pmap.size
assert pmap.select(Nothing()).size == 0

# Accumulate a query over a variable list of strata without a special case.
wanted = ["0-4", "10+"]
query = reduce(operator.or_, (age[name] for name in wanted), Nothing())
assert pmap.select(query).size == 18

## Selectors are validated against the map

A selector is checked when it is resolved, not when it is written. That check
uses the map's registered properties, so a stale trait name or a property that
was never applied is reported with the available alternatives.

In [ ]:
sex = Property("sex", ("female", "male"))

try:
    pmap.select(sex["female"])
except KeyError as exc:
    print("unapplied property :", exc)

try:
    pmap.select(age[("0-4",)] & vax["three-dose"])
except KeyError as exc:
    print("unknown trait      :", exc)

## Query results are cached

Resolving a selector caches the evaluated array on the map, keyed by the
selector value. Because selectors are frozen dataclasses compared by value, an
equal selector built somewhere else hits the same cache entry.

In [ ]:
first = pmap.mask(state["I"] & age["5-9"])
again = pmap.mask(state["I"] & age["5-9"])   # rebuilt, structurally equal

assert first.tolist() == again.tolist()

`copy()` returns the same table with an empty cache — relevant only when a map
is held for a long time and a large number of one-off queries have accumulated.

In [ ]:
fresh = pmap.copy()
assert fresh == pmap

---

Next: {doc}`04-ragged-stratification` covers what happens when a property does
not apply to every compartment — and why `~` alone is not enough there.